In [2]:
# Inspect the maps and the raw column values to fix the mapping direction.
for col, mapping in cat_maps.items():
    items = list(mapping.items())[:3]
    print(f"{col}: {len(mapping)} entries | sample {items}")
print()
for col in cat_maps:
    if col in df.columns:
        print(f"{col}: dtype={df[col].dtype} | sample values {df[col].dropna().unique()[:3]}")

Receiving Currency: 15 entries | sample [('0', 'Australian Dollar'), ('1', 'Bitcoin'), ('2', 'Brazil Real')]
Payment Currency: 15 entries | sample [('0', 'Australian Dollar'), ('1', 'Bitcoin'), ('2', 'Brazil Real')]
Payment Format: 7 entries | sample [('0', 'ACH'), ('1', 'Bitcoin'), ('2', 'Cash')]

Receiving Currency: dtype=object | sample values ['US Dollar' 'Bitcoin' 'Euro']
Payment Currency: dtype=object | sample values ['US Dollar' 'Bitcoin' 'Euro']
Payment Format: dtype=object | sample values ['Reinvestment' 'Cheque' 'Credit Card']


In [3]:
# Maps are stored code -> name; invert to name -> code before applying.
for col, mapping in cat_maps.items():
    if col in df.columns and df[col].dtype == object:
        inv = {v: int(k) for k, v in mapping.items()}
        mapped = df[col].map(inv)
        assert mapped.notna().all(), f"{col}: {mapped.isna().sum()} unmapped values"
        df[col] = mapped.astype(int)
print("category columns encoded:", [c for c in cat_maps if c in df.columns])

category columns encoded: ['Receiving Currency', 'Payment Currency', 'Payment Format']


In [4]:
# Re-score the test rows with the sliced boosters. Gate: diffs must be ~0.
m1 = xgb.Booster(); m1.load_model(MODELS/"m1_baseline.json")
m2 = xgb.Booster(); m2.load_model(MODELS/"m2_graph_xgb.json")
n1, n2 = thr1["n_trees_used"], thr2["n_trees_used"]      # 482 / 430

probs1 = pd.read_parquet(MODELS/"m1_test_probs.parquet")
probs2 = pd.read_parquet(MODELS/"m2_test_probs.parquet")
test_idx = probs1.index
X1 = df.loc[test_idx, m1_feats]
X2 = df.loc[test_idx, m2_feats]

p1_new = m1.predict(xgb.DMatrix(X1), iteration_range=(0, n1))
p2_new = m2.predict(xgb.DMatrix(X2), iteration_range=(0, n2))
print("trees used:", n1, n2)
print("M1 max abs diff:", np.abs(p1_new - probs1["m1_prob"].values).max())
print("M2 max abs diff:", np.abs(p2_new - probs2["m2_prob"].values).max())
print("is_main rows:", int(probs1["is_main"].sum()))

trees used: 482 430
M1 max abs diff: 0.0
M2 max abs diff: 0.0
is_main rows: 760531


In [5]:
# Ranking stability across sample and background sizes. Compares each configuration's
# mean-|SHAP| ordering against the registered run (n=10,000, background=1,000).
main_idx = probs1.index[probs1["is_main"].values]
y = df.loc[main_idx, "Is Laundering"].astype(int).values

# Background: stratified draw from TRAIN rows, matching Notebook 08's protocol.
train_idx = df.index[df["split"] == "train"] if "split" in df.columns else None
print("train rows available:", None if train_idx is None else len(train_idx))

def stratified_bg(n_bg, seed=SEED):
    r = np.random.default_rng(seed)
    tr_y = df.loc[train_idx, "Is Laundering"].astype(int).values
    pos, neg = train_idx[tr_y == 1], train_idx[tr_y == 0]
    n_pos = max(1, int(round(n_bg * len(pos) / len(train_idx))))
    return np.concatenate([r.choice(pos, n_pos, replace=False),
                           r.choice(neg, n_bg - n_pos, replace=False)])

def mean_abs_shap(booster, feats, n_trees, rows, bg_rows):
    ex = shap.TreeExplainer(booster[:n_trees], data=df.loc[bg_rows, feats],
                            feature_perturbation="interventional", model_output="raw")
    sv = ex.shap_values(df.loc[rows, feats], check_additivity=False)
    return pd.Series(np.abs(sv).mean(axis=0), index=feats)

configs = [(5000, 1000), (10000, 1000), (20000, 1000), (10000, 500), (10000, 2000)]
results = {}
for n_s, n_bg in configs:
    r = np.random.default_rng(SEED)
    rows = r.choice(main_idx, n_s, replace=False)
    bg = stratified_bg(n_bg)
    results[(n_s, n_bg)] = {
        "m1": mean_abs_shap(m1, m1_feats, n1, rows, bg),
        "m2": mean_abs_shap(m2, m2_feats, n2, rows, bg),
    }
    print(f"done: n={n_s}, bg={n_bg}")

train rows available: 3554957


 99%|===================| 4969/5000 [00:50<00:00]        

done: n=5000, bg=1000


 99%|===================| 9925/10000 [01:58<00:00]        

done: n=10000, bg=1000


...

done: n=20000, bg=1000


 99%|===================| 9911/10000 [01:40<00:00]        

done: n=10000, bg=500


100%|===================| 9970/10000 [01:42<00:00]        

done: n=10000, bg=2000


In [6]:
# How much does the ranking move across configurations? Reference = registered run.
REF = (10000, 1000)

def rbo(l1, l2, p=0.9):
    s1, s2, agree, total = set(), set(), 0.0, 0.0
    for d in range(1, max(len(l1), len(l2)) + 1):
        if d <= len(l1): s1.add(l1[d-1])
        if d <= len(l2): s2.add(l2[d-1])
        w = p ** (d-1)
        agree += w * (len(s1 & s2) / d); total += w
    return agree / total

rows_out = []
for key, res in results.items():
    for arm in ("m1", "m2"):
        ref = results[REF][arm]
        cur = res[arm]
        L_ref = ref.sort_values(ascending=False).index.tolist()
        L_cur = cur.sort_values(ascending=False).index.tolist()
        top5 = len(set(L_ref[:5]) & set(L_cur[:5]))
        rows_out.append({
            "n_sample": key[0], "n_background": key[1], "model": arm.upper(),
            "spearman_vs_ref": spearmanr(ref, cur.reindex(ref.index)).statistic,
            "rbo_vs_ref": rbo(L_ref, L_cur),
            "top5_overlap": f"{top5}/5",
            "top1": L_cur[0],
        })

stab = pd.DataFrame(rows_out).sort_values(["model", "n_sample", "n_background"])
print(stab.to_string(index=False))
stab.to_csv(TABLES / "11_ranking_stability.csv", index=False)
print("\nwritten:", TABLES / "11_ranking_stability.csv")

 n_sample  n_background model  spearman_vs_ref  rbo_vs_ref top5_overlap           top1
     5000          1000    M1         0.987879    0.990821          5/5 Payment Format
    10000           500    M1         1.000000    1.000000          5/5 Payment Format
    10000          1000    M1         1.000000    1.000000          5/5 Payment Format
    10000          2000    M1         0.987879    0.988344          5/5 Payment Format
    20000          1000    M1         0.987879    0.990821          5/5 Payment Format
     5000          1000    M2         0.998632    0.992946          5/5 Payment Format
    10000           500    M2         0.989060    0.911453          4/5 Payment Format
    10000          1000    M2         1.000000    1.000000          5/5 Payment Format
    10000          2000    M2         0.995214    0.933676          5/5 Payment Format
    20000          1000    M2         0.998632    0.992946          5/5 Payment Format

written: c:\Fintech-Project\graph_AML_pipe